In [1]:
import pandas as pd
from pathlib import Path

In [2]:
df = pd.read_parquet("hf://datasets/ruslanmv/ai-medical-chatbot/dialogues.parquet")
df.head()

/home/calvotom/anaconda3/envs/analyse_exploration_data/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,Description,Patient,Doctor
0,Q. What does abutment of the nerve root mean?,"Hi doctor,I am just wondering what is abutting...",Hi. I have gone through your query with dilige...
1,Q. What should I do to reduce my weight gained...,"Hi doctor, I am a 22-year-old female who was d...",Hi. You have really done well with the hypothy...
2,Q. I have started to get lots of acne on my fa...,Hi doctor! I used to have clear skin but since...,Hi there Acne has multifactorial etiology. Onl...
3,Q. Why do I have uncomfortable feeling between...,"Hello doctor,I am having an uncomfortable feel...",Hello. The popping and discomfort what you fel...
4,Q. My symptoms after intercourse threatns me e...,"Hello doctor,Before two years had sex with a c...",Hello. The HIV test uses a finger prick blood ...


In [3]:
import os
os.makedirs('../../medical_dataset/raw', exist_ok=True)

df.to_parquet('../../medical_dataset/raw/dialogues.parquet')

In [4]:
# Structure générale
print("Shape:", df.shape)
print("\nColonnes:", df.columns.tolist())
print("\nTypes:")
print(df.dtypes)

# Aperçu
df.head()

Shape: (256916, 3)

Colonnes: ['Description', 'Patient', 'Doctor']

Types:
Description    str
Patient        str
Doctor         str
dtype: object


,Description,Patient,Doctor
0,Q. What does abutment of the nerve root mean?,"Hi doctor,I am just wondering what is abutting...",Hi. I have gone through your query with dilige...
1,Q. What should I do to reduce my weight gained...,"Hi doctor, I am a 22-year-old female who was d...",Hi. You have really done well with the hypothy...
2,Q. I have started to get lots of acne on my fa...,Hi doctor! I used to have clear skin but since...,Hi there Acne has multifactorial etiology. Onl...
3,Q. Why do I have uncomfortable feeling between...,"Hello doctor,I am having an uncomfortable feel...",Hello. The popping and discomfort what you fel...
4,Q. My symptoms after intercourse threatns me e...,"Hello doctor,Before two years had sex with a c...",Hello. The HIV test uses a finger prick blood ...


In [5]:
# Valeurs manquantes
print("Valeurs manquantes par colonne:")
print(df.isnull().sum())
print("\n% manquant:")
print((df.isnull().sum() / len(df) * 100).round(2))

Valeurs manquantes par colonne:
Description    0
Patient        0
Doctor         0
dtype: int64

% manquant:
Description    0.0
Patient        0.0
Doctor         0.0
dtype: float64


In [6]:
# Doublons
print("Nombre de lignes dupliquées (toutes colonnes):", df.duplicated().sum())

Nombre de lignes dupliquées (toutes colonnes): 10378


In [7]:

for col in df.select_dtypes(include='object').columns:
    print(f"\n--- {col} ---")
    print(df[col].str.len().describe())


--- Description ---
count    256916.000000
mean         59.445912
std          22.295816
min           1.000000
25%          44.000000
50%          56.000000
75%          70.000000
max        1503.000000
Name: Description, dtype: float64

--- Patient ---
count    256916.000000
mean        436.463945
std         299.354642
min           1.000000
25%         283.000000
50%         353.000000
75%         491.000000
max       17735.000000
Name: Patient, dtype: float64

--- Doctor ---
count    256916.000000
mean        537.439805
std         338.735951
min           2.000000
25%         318.000000
50%         475.000000
75%         675.000000
max       11385.000000
Name: Doctor, dtype: float64


/tmp/ipykernel_91980/1580540974.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='object').columns:


In [8]:

df_clean = df.drop_duplicates().reset_index(drop=True)
print("Après suppression doublons:", df_clean.shape)

Après suppression doublons: (246538, 3)


In [9]:
df['len_patient'] = df['Patient'].str.len()
df['len_doctor'] = df['Doctor'].str.len()
df['len_description'] = df['Description'].str.len()

# Top 5 plus longs
print(df.nlargest(5, 'len_patient')[['Patient', 'len_patient']])
print(df.nlargest(5, 'len_doctor')[['Doctor', 'len_doctor']])

                                                  Patient  len_patient
181031  this is very long I was diagnosed with HSV 2 a...        17735
40219   Our married daughter in her early 40 s lives i...        11188
35466   Hi there, to anyone out there who can help?? T...        10687
35110   MEDICAL HISTORY Dear Doctor, I am writing this...         9782
45884   Mr Obiedullah Baig +(91)-44-0000 My names are ...         9536
                                                   Doctor  len_doctor
92030   HiThe first step in making the diagnosis of a ...       11385
134696  Hi welcome to HCMI have gone thru your query r...        9239
134303  HiDisoriented feeling while on Codeine and Mor...        8930
236481  hi and pleased to answer youHeart failure is a...        8195
92032   HiEczema is often used as an umbrella term for...        6518


In [10]:
print(df[df['len_patient'] < 10][['Patient', 'len_patient']].head(10))
print(df[df['len_doctor'] < 10][['Doctor', 'len_doctor']].head(10))

        Patient  len_patient
8669     Hello,            6
43821      help            4
43877  go oline            8
73848      thtr            4
73886   zzxczxc            7
73887         .            1
73890      SDfc            4
73901         k            1
73947         j            1
81156  aesthema            8
       Doctor  len_doctor
2894      Hi.           3
3952   Hello.           6
4071   Hello.           6
17550     Hi.           3
21190     Hi.           3
21233     Hi.           3
26474     Hi.           3
27240     Hi.           3
29057      Hi           2
29550     Hi.           3


In [11]:
MIN_LEN_PATIENT = 15
MIN_LEN_DOCTOR = 15
MAX_LEN = 3000   

df_clean = df[
    (df['len_patient'] >= MIN_LEN_PATIENT) &
    (df['len_doctor'] >= MIN_LEN_DOCTOR) &
    (df['len_patient'] <= MAX_LEN) &
    (df['len_doctor'] <= MAX_LEN) &
    (df['len_description'] >= 5)
].copy()

print(f"Avant: {len(df)} | Après: {len(df_clean)} | Perte: {len(df)-len(df_clean)} ({(len(df)-len(df_clean))/len(df)*100:.1f}%)")

Avant: 256916 | Après: 256333 | Perte: 583 (0.2%)


In [12]:
import re

# Recherche de numéros de téléphone / emails dans les colonnes
phone_pattern = r'\+?\d{1,3}[-.\s]?\(?\d{2,4}\)?[-.\s]?\d{3,4}[-.\s]?\d{3,4}'
email_pattern = r'[\w\.-]+@[\w\.-]+\.\w+'

for col in ['Patient', 'Doctor', 'Description']:
    n_phone = df_clean[col].str.contains(phone_pattern, regex=True, na=False).sum()
    n_email = df_clean[col].str.contains(email_pattern, regex=True, na=False).sum()
    print(f"{col}: {n_phone} possibles téléphones, {n_email} possibles emails")

Patient: 154 possibles téléphones, 82 possibles emails
Doctor: 225 possibles téléphones, 2231 possibles emails
Description: 1 possibles téléphones, 0 possibles emails


In [13]:
# Exemples de "téléphones" détectés
matches_phone = df_clean[df_clean['Doctor'].str.contains(phone_pattern, regex=True, na=False)]
print(matches_phone['Doctor'].head(10).tolist())

['Hi. I have analyzed the symptoms you described in the query. The appearance and pattern of aura are actually suggestive of the fact that the electrical activity of your brain is out of control. Though it is not a full blown seizure, it is surely an early phase of seizures. I suggest you increase the dose of Tegretol (Carbamazepine) to 200-200-400. It would be best if you can repeat the MRI brain whenever the pattern of seizures changes. There is a possibility that something might have changed in brain structure also. When you go for MRI, please get an MRI brain epilepsy protocol. This protocol studies the medial temporal lobe region of the brain in detail. The way your seizures are; most likely they are coming from the medial temporal lobes. I hope my answer helps you. For more information consult a neurologist online -->', 'Dear Priya:I would like to give you following suggestions. Kindly follow this religiously for 21 days continuously and I am sure you would be much better and wou

In [14]:
# Exemples d'"emails" détectés
matches_email = df_clean[df_clean['Doctor'].str.contains(email_pattern, regex=True, na=False)]
print(matches_email['Doctor'].head(10).tolist())

["Hi,  The definative treatment of phimosis is circumscision .Masturbation is not the solution for it n later it may led to loss of libido , premature ejaculation , etc.  Just talk with your parent regarding phimosis (don't feel ashamed as plenty of people suffer from it and leaving it like that may lead to other complication in your life) and visit a urologist for further management .HOPE THIS WILL HELP THANKS N REGARDSdr.kishorekunal@gmail.com", "HelloI can understand the stress you both are going through. Stress is a very common cause of erection problem. Please specify for viagra and cialis which dosage you have used . Some need higher dosages. You should also consider smoking, alcoholism,obesity, diabetes, hypertension as possible cause for erection problem so get ECG,Blood sugar, Lipid Profile, and morning testosterone level between 8-9 AM. IVF can help you with getting a baby but not in sex life. you may wait for a couple of months and see your sex life gets better. If you don't

In [15]:
phone_pattern = r'\+?\d{1,3}[-.\s]?\(?\d{2,4}\)?[-.\s]?\d{3,4}[-.\s]?\d{3,4}'
email_pattern = r'[\w\.-]+@[\w\.-]+\.\w+'

def anonymize(text):
    if pd.isna(text):
        return text
    text = re.sub(email_pattern, '[EMAIL]', text)
    text = re.sub(phone_pattern, '[PHONE]', text)
    return text

df_clean['Patient'] = df_clean['Patient'].apply(anonymize)
df_clean['Doctor'] = df_clean['Doctor'].apply(anonymize)
df_clean['Description'] = df_clean['Description'].apply(anonymize)

In [16]:
for col in ['Patient', 'Doctor', 'Description']:
    n_phone = df_clean[col].str.contains(phone_pattern, regex=True, na=False).sum()
    n_email = df_clean[col].str.contains(email_pattern, regex=True, na=False).sum()
    print(f"{col}: {n_phone} téléphones restants, {n_email} emails restants")

Patient: 0 téléphones restants, 0 emails restants
Doctor: 0 téléphones restants, 0 emails restants
Description: 0 téléphones restants, 0 emails restants


In [17]:
print("Avant doublons:", df_clean.shape)
df_clean = df_clean.drop_duplicates(subset=['Patient', 'Doctor', 'Description'])
print("Après doublons:", df_clean.shape)

Avant doublons: (256333, 6)
Après doublons: (245955, 6)


In [18]:
df_clean['len_patient'] = df_clean['Patient'].str.len()
df_clean['len_doctor'] = df_clean['Doctor'].str.len()
df_clean['len_description'] = df_clean['Description'].str.len()

MIN_LEN_PATIENT = 15
MIN_LEN_DOCTOR = 15
MAX_LEN = 3000

before = len(df_clean)
df_clean = df_clean[
    (df_clean['len_patient'] >= MIN_LEN_PATIENT) &
    (df_clean['len_doctor'] >= MIN_LEN_DOCTOR) &
    (df_clean['len_patient'] <= MAX_LEN) &
    (df_clean['len_doctor'] <= MAX_LEN) &
    (df_clean['len_description'] >= 5)
].copy()

print(f"Avant: {before} | Après: {len(df_clean)} | Perte: {before-len(df_clean)} ({(before-len(df_clean))/before*100:.1f}%)")

Avant: 245955 | Après: 245955 | Perte: 0 (0.0%)


In [19]:
df_clean = df_clean.drop(columns=['len_patient', 'len_doctor', 'len_description'])
df_clean = df_clean.reset_index(drop=True)
print(df_clean.shape)
df_clean.head()

(245955, 3)


,Description,Patient,Doctor
0,Q. What does abutment of the nerve root mean?,"Hi doctor,I am just wondering what is abutting...",Hi. I have gone through your query with dilige...
1,Q. What should I do to reduce my weight gained...,"Hi doctor, I am a 22-year-old female who was d...",Hi. You have really done well with the hypothy...
2,Q. I have started to get lots of acne on my fa...,Hi doctor! I used to have clear skin but since...,Hi there Acne has multifactorial etiology. Onl...
3,Q. Why do I have uncomfortable feeling between...,"Hello doctor,I am having an uncomfortable feel...",Hello. The popping and discomfort what you fel...
4,Q. My symptoms after intercourse threatns me e...,"Hello doctor,Before two years had sex with a c...",Hello. The HIV test uses a finger prick blood ...


In [20]:
print("Min longueur Patient:", df_clean['Patient'].str.len().min())
print("Min longueur Doctor:", df_clean['Doctor'].str.len().min())
print("Max longueur Patient:", df_clean['Patient'].str.len().max())
print("Max longueur Doctor:", df_clean['Doctor'].str.len().max())

Min longueur Patient: 15
Min longueur Doctor: 15
Max longueur Patient: 2999
Max longueur Doctor: 3000


## Rapport de Qualité :Dataset Médical (ruslanmv/ai-medical-chatbot)

### 1. Vue d'ensemble du dataset brut

| Caractéristique | Valeur |
|---|---|
| Lignes initiales | 256 916 |
| Colonnes | `Description`, `Patient`, `Doctor` |
| Valeurs manquantes | 0 (toutes colonnes) |
| Doublons exacts | 10 378 (4.0%) |

Le dataset ne présentait aucune valeur manquante, mais une analyse plus poussée a révélé plusieurs problèmes de qualité non visibles au premier abord.

### 2. Problèmes identifiés

**a) Doublons**
10 961 lignes dupliquées (4.3% du dataset) ont été détectées sur l'ensemble des trois colonnes, probablement issues d'un scraping multiple ou d'une agrégation de sources redondantes.

**b) Textes dégénérés (trop courts)**
Des entrées quasi vides ou inexploitables ont été identifiées côté `Patient` ("help", "thtr", "zzxczxc", ".") et côté `Doctor` ("Hi.", "Hello.")ces dernières correspondant probablement à des réponses tronquées lors de la collecte des données.

**c) Outliers de longueur (trop longs)**
Certaines entrées dépassaient 17 000 caractères (max observé : 17 735 pour `Patient`, 11 385 pour `Doctor`), très éloignées de la médiane (~353 et ~475 caractères respectivement). Ces textes très longs auraient déséquilibré l'apprentissage du modèle lors du fine-tuning.

**d) Données personnelles identifiables **
Un nombre significatif de réponses médecins contenaient des informations personnelles : numéros de téléphone (154 dans `Patient`, 225 dans `Doctor`) et adresses email (82 dans `Patient`, **2231** dans `Doctor`). Plusieurs cas incluaient des coordonnées complètes de praticiens (nom, téléphone, parfois adresse physique), posant un risque de confidentialité et de conformité (RGPD) si utilisées telles quelles pour l'entraînement.

### 3. Actions de nettoyage appliquées

| Action | Détail |
|---|---|
| Anonymisation | Remplacement des emails/téléphones par `[EMAIL]` / `[PHONE]` (regex) |
| Dédoublonnage | Suppression des lignes strictement identiques |
| Filtrage longueur | Conservation des textes entre 15 et 3000 caractères (`Patient`/`Doctor`), ≥5 caractères (`Description`) |

### 4. Résultat final

| Étape | Lignes restantes | Perte cumulée |
|---|---|---|
| Brut | 256 916 | - |
| Après dédoublonnage | 245 955 | 4.3% |
| Après filtrage longueur | 245 955 | 0% (déjà éliminés via dédoublonnage) |
| **Total conservé** | **245 955** | **4.3%** |


In [23]:
df_clean.to_parquet('../../medical_dataset/interim/dialogues_clean.parquet')
print("Sauvegardé :", df_clean.shape)

Sauvegardé : (245955, 3)
